In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv("/content/DATA_Havvy Rain.CSV")

In [ ]:
print(df.shape)
print(df.head())
print(df.info())
print(df.describe())

(4254, 20)
   Time(ms)  Temp(C)  Humidity(%)  RainAnalog  RainDigital  SNR1  SNR2  SNR3  \
0      2088     21.8         92.0          46            1    31    27    28   
1      4080     21.9         92.9          45            1    27    27    28   
2      6074     22.0         92.0          40            1    27    27    27   
3      8074     22.1         91.6          30            1    27    27    27   
4     10077     21.9         92.0          31            1     3    26    26   

   SNR4  SNR5  SNR6  SNR7  SNR8  SNR9  SNR10  SNR11  SNR12  SNR13  SNR14  \
0    27    32    27    28     0     0      0      0      0      0      0   
1    27    32    27    27     0     0      0      0      0      0      0   
2    27    32    27    27     0     0      0      0      0      0      0   
3    27    31    26    26    26     0      0      0      0      0      0   
4    26    31    25    26    26     0      0      0      0      0      0   

   SNR15  
0      0  
1      0  
2      0  
3      

In [ ]:
df.isnull().sum()
(df == 0).sum()

,0
Time(ms),0
Temp(C),0
Humidity(%),0
RainAnalog,0
RainDigital,0
SNR1,195
SNR2,231
SNR3,306
SNR4,329
SNR5,415


In [ ]:
snr_cols = [col for col in df.columns if 'SNR' in col]

for col in snr_cols:
    df[col] = df[col].apply(lambda x: np.nan if x > 100 else x)

In [ ]:
for col in snr_cols:
    df[col] = df[col].replace(0, np.nan)

In [ ]:
df[snr_cols] = df[snr_cols].interpolate(method='linear')

In [ ]:
df[snr_cols] = df[snr_cols].fillna(method='bfill')
df[snr_cols] = df[snr_cols].fillna(method='ffill')

/tmp/ipython-input-3485316465.py:1: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df[snr_cols] = df[snr_cols].fillna(method='bfill')
/tmp/ipython-input-3485316465.py:2: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df[snr_cols] = df[snr_cols].fillna(method='ffill')


In [ ]:
for col in snr_cols:
    df[col] = df[col].rolling(window=5, min_periods=1).mean()

In [ ]:
df['Mean_SNR'] = df[snr_cols].mean(axis=1)

In [ ]:
df['SNR_Var'] = df[snr_cols].var(axis=1)

In [ ]:
df['Sat_Count'] = df[snr_cols].notnull().sum(axis=1)

In [ ]:
df[['Temp(C)', 'Humidity(%)']] = df[['Temp(C)', 'Humidity(%)']].interpolate()

In [ ]:
for col in snr_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df[col] = np.clip(df[col], lower, upper)

In [ ]:
from sklearn.preprocessing import RobustScaler

features = df.drop(['RainDigital'], axis=1)

scaler = RobustScaler()
scaled = scaler.fit_transform(features)

X = pd.DataFrame(scaled, columns=features.columns)
y = df['RainDigital']

In [ ]:
snr_cols = [col for col in df.columns if "SNR" in col]

for col in snr_cols:
    df.loc[df[col] > 100, col] = np.nan

In [ ]:
for col in snr_cols:
    df.loc[df[col] == 0, col] = np.nan

In [ ]:
df[['Temp(C)', 'Humidity(%)']] = df[['Temp(C)', 'Humidity(%)']].interpolate()

In [ ]:
for col in snr_cols:
    df[col] = df[col].rolling(window=5, min_periods=1).mean()

In [ ]:
for col in snr_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df[col] = df[col].clip(lower, upper)

In [ ]:
df['Mean_SNR'] = df[snr_cols].mean(axis=1)
df['SNR_Var'] = df[snr_cols].var(axis=1)
df['Sat_Count'] = df[snr_cols].notna().sum(axis=1)

In [ ]:
snr_cols = [col for col in df.columns if "SNR" in col]

In [ ]:
df[snr_cols] = df[snr_cols].round()

In [ ]:
df[snr_cols] = df[snr_cols].astype("Int64")

In [ ]:
print(df[snr_cols].head())
print(df[snr_cols].dtypes)

   SNR1  SNR2  SNR3  SNR4  SNR5  SNR6  SNR7  SNR8  SNR9  SNR10  SNR11  SNR12  \
0    31    27    28    27    32    27    28    26    40     22     29     64   
1    30    27    28    27    32    27    28    26    40     22     29     64   
2    29    27    28    27    32    27    28    26    40     22     29     64   
3    29    27    28    27    32    27    27    26    40     22     29     64   
4    28    27    28    27    32    27    27    26    40     22     29     64   

   SNR13  SNR14  SNR15  Mean_SNR  SNR_Var  
0     24     30     25        31       95  
1     24     30     25        31       96  
2     24     30     25        31       96  
3     24     30     25        30       96  
4     24     30     25        30       96  
SNR1        Int64
SNR2        Int64
SNR3        Int64
SNR4        Int64
SNR5        Int64
SNR6        Int64
SNR7        Int64
SNR8        Int64
SNR9        Int64
SNR10       Int64
SNR11       Int64
SNR12       Int64
SNR13       Int64
SNR14       Int64
SNR

In [ ]:
df.to_csv("Rainy_Havvy_Clean.csv", index=False)